In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("set1").getOrCreate()



# 1.Data Ingestion & Schema Analysis


In [0]:
# Q1.1 - Load CSV using PySpark with schema inference
df = spark.read.option("header", True).option("inferSchema", True).csv("file:/Workspace/Shared/traffic_logs.csv")
df.printSchema()
df.show()


root
 |-- LogID: string (nullable = true)
 |-- VehicleID: string (nullable = true)
 |-- EntryPoint: string (nullable = true)
 |-- ExitPoint: string (nullable = true)
 |-- EntryTime: timestamp (nullable = true)
 |-- ExitTime: timestamp (nullable = true)
 |-- VehicleType: string (nullable = true)
 |-- SpeedKMH: integer (nullable = true)
 |-- TollPaid: integer (nullable = true)

+-----+---------+----------+---------+-------------------+-------------------+-----------+--------+--------+
|LogID|VehicleID|EntryPoint|ExitPoint|          EntryTime|           ExitTime|VehicleType|SpeedKMH|TollPaid|
+-----+---------+----------+---------+-------------------+-------------------+-----------+--------+--------+
| L001|     V001|     GateA|    GateC|2024-05-01 08:01:00|2024-05-01 08:20:00|        Car|      60|      50|
| L002|     V002|     GateB|    GateC|2024-05-01 08:10:00|2024-05-01 08:45:00|      Truck|      45|     100|
| L003|     V003|     GateA|    GateD|2024-05-01 09:00:00|2024-05-01 09:18:0

In [0]:
# Q1.2 - Manually define schema and compare
from pyspark.sql.types import *

schema = StructType([
    StructField("LogID", StringType()),
    StructField("VehicleID", StringType()),
    StructField("EntryPoint", StringType()),
    StructField("ExitPoint", StringType()),
    StructField("EntryTime", TimestampType()),
    StructField("ExitTime", TimestampType()),
    StructField("VehicleType", StringType()),
    StructField("SpeedKMH", IntegerType()),
    StructField("TollPaid", IntegerType())
])

df_manual = spark.read.option("header", True).schema(schema).csv("file:/Workspace/Shared/traffic_logs.csv")
df_manual.printSchema()
df_manual.show()


root
 |-- LogID: string (nullable = true)
 |-- VehicleID: string (nullable = true)
 |-- EntryPoint: string (nullable = true)
 |-- ExitPoint: string (nullable = true)
 |-- EntryTime: timestamp (nullable = true)
 |-- ExitTime: timestamp (nullable = true)
 |-- VehicleType: string (nullable = true)
 |-- SpeedKMH: integer (nullable = true)
 |-- TollPaid: integer (nullable = true)

+-----+---------+----------+---------+-------------------+-------------------+-----------+--------+--------+
|LogID|VehicleID|EntryPoint|ExitPoint|          EntryTime|           ExitTime|VehicleType|SpeedKMH|TollPaid|
+-----+---------+----------+---------+-------------------+-------------------+-----------+--------+--------+
| L001|     V001|     GateA|    GateC|2024-05-01 08:01:00|2024-05-01 08:20:00|        Car|      60|      50|
| L002|     V002|     GateB|    GateC|2024-05-01 08:10:00|2024-05-01 08:45:00|      Truck|      45|     100|
| L003|     V003|     GateA|    GateD|2024-05-01 09:00:00|2024-05-01 09:18:0

# 2. Derived Column Creation

In [0]:
# Q2 - Derived Columns: TripDurationMinutes and IsOverspeed
from pyspark.sql.functions import col, unix_timestamp, when

df2 = df_manual.withColumn("TripDurationMinutes", 
                           (unix_timestamp("ExitTime") - unix_timestamp("EntryTime")) / 60)\
               .withColumn("IsOverspeed", col("SpeedKMH") > 60)
df2.select("LogID", "TripDurationMinutes", "IsOverspeed").show()


+-----+-------------------+-----------+
|LogID|TripDurationMinutes|IsOverspeed|
+-----+-------------------+-----------+
| L001|               19.0|      false|
| L002|               35.0|      false|
| L003|               18.0|      false|
| L004|               20.0|       true|
| L005|               35.0|      false|
+-----+-------------------+-----------+



# 3.Vehicle Behavior Aggregations

In [0]:
# Q3.1 - Average speed per VehicleType
from pyspark.sql.functions import avg

df2.groupBy("VehicleType").agg(avg("SpeedKMH").alias("AvgSpeed")).show()


+-----------+--------+
|VehicleType|AvgSpeed|
+-----------+--------+
|       Bike|    55.0|
|        Car|    70.0|
|      Truck|    45.0|
|        Bus|    40.0|
+-----------+--------+



In [0]:
# Q3.2 - Total toll collected per EntryPoint
from pyspark.sql.functions import sum

df2.groupBy("EntryPoint").agg(sum("TollPaid").alias("TotalToll")).show()


+----------+---------+
|EntryPoint|TotalToll|
+----------+---------+
|     GateA|       80|
|     GateB|      170|
|     GateC|       50|
+----------+---------+



In [0]:
# Q3.3 - Most used ExitPoint
from pyspark.sql.functions import count

df2.groupBy("ExitPoint").agg(count("*").alias("Usage")).orderBy(col("Usage").desc()).show(1)


+---------+-----+
|ExitPoint|Usage|
+---------+-----+
|    GateD|    2|
+---------+-----+
only showing top 1 row



# 4.Window Functions

In [0]:
# Q4.1 - Rank vehicles by speed within VehicleType
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

w = Window.partitionBy("VehicleType").orderBy(col("SpeedKMH").desc())
df2.withColumn("SpeedRank", rank().over(w)).select("VehicleID", "VehicleType", "SpeedKMH", "SpeedRank").show()


+---------+-----------+--------+---------+
|VehicleID|VehicleType|SpeedKMH|SpeedRank|
+---------+-----------+--------+---------+
|     V003|       Bike|      55|        1|
|     V005|        Bus|      40|        1|
|     V004|        Car|      80|        1|
|     V001|        Car|      60|        2|
|     V002|      Truck|      45|        1|
+---------+-----------+--------+---------+



In [0]:
# Q4.2 - Find last exit time for each vehicle using lag()
from pyspark.sql.functions import lag

w2 = Window.partitionBy("VehicleID").orderBy("ExitTime")
df2.withColumn("PrevExitTime", lag("ExitTime").over(w2)).select("VehicleID", "ExitTime", "PrevExitTime").show()


+---------+-------------------+------------+
|VehicleID|           ExitTime|PrevExitTime|
+---------+-------------------+------------+
|     V001|2024-05-01 08:20:00|        NULL|
|     V002|2024-05-01 08:45:00|        NULL|
|     V003|2024-05-01 09:18:00|        NULL|
|     V004|2024-05-01 09:35:00|        NULL|
|     V005|2024-05-01 10:40:00|        NULL|
+---------+-------------------+------------+



# 5.Session Segmentation


In [0]:
# Q5 - Session Segmentation: Group by VehicleID, find idle time between trips
from pyspark.sql.functions import unix_timestamp

df_sessions = df2.withColumn("PrevExitTime", lag("ExitTime").over(w2))
df_sessions = df_sessions.withColumn("IdleMinutes", 
                     (unix_timestamp("EntryTime") - unix_timestamp("PrevExitTime")) / 60)
df_sessions.select("VehicleID", "EntryTime", "ExitTime", "PrevExitTime", "IdleMinutes").show()


+---------+-------------------+-------------------+------------+-----------+
|VehicleID|          EntryTime|           ExitTime|PrevExitTime|IdleMinutes|
+---------+-------------------+-------------------+------------+-----------+
|     V001|2024-05-01 08:01:00|2024-05-01 08:20:00|        NULL|       NULL|
|     V002|2024-05-01 08:10:00|2024-05-01 08:45:00|        NULL|       NULL|
|     V003|2024-05-01 09:00:00|2024-05-01 09:18:00|        NULL|       NULL|
|     V004|2024-05-01 09:15:00|2024-05-01 09:35:00|        NULL|       NULL|
|     V005|2024-05-01 10:05:00|2024-05-01 10:40:00|        NULL|       NULL|
+---------+-------------------+-------------------+------------+-----------+



# 6.Anomaly Detection

In [0]:
# Q6.1 - Anomaly Detection: speed > 70 and TripDuration < 10
df2.filter((col("SpeedKMH") > 70) & (col("TripDurationMinutes") < 10)).show()


+-----+---------+----------+---------+---------+--------+-----------+--------+--------+-------------------+-----------+
|LogID|VehicleID|EntryPoint|ExitPoint|EntryTime|ExitTime|VehicleType|SpeedKMH|TollPaid|TripDurationMinutes|IsOverspeed|
+-----+---------+----------+---------+---------+--------+-----------+--------+--------+-------------------+-----------+
+-----+---------+----------+---------+---------+--------+-----------+--------+--------+-------------------+-----------+



In [0]:
# Q6.2 - Paid less toll for longer trips
df2.orderBy(col("TripDurationMinutes").desc(), col("TollPaid").asc()).show()


+-----+---------+----------+---------+-------------------+-------------------+-----------+--------+--------+-------------------+-----------+
|LogID|VehicleID|EntryPoint|ExitPoint|          EntryTime|           ExitTime|VehicleType|SpeedKMH|TollPaid|TripDurationMinutes|IsOverspeed|
+-----+---------+----------+---------+-------------------+-------------------+-----------+--------+--------+-------------------+-----------+
| L005|     V005|     GateB|    GateA|2024-05-01 10:05:00|2024-05-01 10:40:00|        Bus|      40|      70|               35.0|      false|
| L002|     V002|     GateB|    GateC|2024-05-01 08:10:00|2024-05-01 08:45:00|      Truck|      45|     100|               35.0|      false|
| L004|     V004|     GateC|    GateD|2024-05-01 09:15:00|2024-05-01 09:35:00|        Car|      80|      50|               20.0|       true|
| L001|     V001|     GateA|    GateC|2024-05-01 08:01:00|2024-05-01 08:20:00|        Car|      60|      50|               19.0|      false|
| L003|     V

In [0]:
# Q6.3 - Suspicious backtracking (ExitPoint earlier than EntryPoint)
df2.filter(col("EntryPoint") > col("ExitPoint")).show()


+-----+---------+----------+---------+-------------------+-------------------+-----------+--------+--------+-------------------+-----------+
|LogID|VehicleID|EntryPoint|ExitPoint|          EntryTime|           ExitTime|VehicleType|SpeedKMH|TollPaid|TripDurationMinutes|IsOverspeed|
+-----+---------+----------+---------+-------------------+-------------------+-----------+--------+--------+-------------------+-----------+
| L005|     V005|     GateB|    GateA|2024-05-01 10:05:00|2024-05-01 10:40:00|        Bus|      40|      70|               35.0|      false|
+-----+---------+----------+---------+-------------------+-------------------+-----------+--------+--------+-------------------+-----------+



# 7.Join with Metadata


In [0]:
# Q7 - Join with Metadata: vehicle_registry.csv
registry = spark.read.option("header", True).option("inferSchema", True).csv("file:/Workspace/Shared/vehicle_registry.csv")

df_joined = df2.join(registry, on="VehicleID", how="inner")
df_joined.groupBy("RegisteredCity").count().show()


+--------------+-----+
|RegisteredCity|count|
+--------------+-----+
|     Bangalore|    1|
|       Chennai|    1|
|        Mumbai|    1|
|          Pune|    1|
|         Delhi|    1|
+--------------+-----+



# 8.Delta Lake Features

In [0]:
# Q8.1 - Save as Delta Table
df2.write.format("delta").mode("overwrite").save("/mnt/delta/traffic_logs")


In [0]:
# Q8.2 - MERGE INTO to update toll for Bikes
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, "/mnt/delta/traffic_logs")

deltaTable.alias("t").merge(
    df2.filter("VehicleType = 'Bike'").alias("s"),
    "t.LogID = s.LogID"
).whenMatchedUpdate(set={"TollPaid": "s.TollPaid + 20"}).execute()


In [0]:
# Q8.3 - Delete trips > 60 minutes
deltaTable.delete("TripDurationMinutes > 60")


In [0]:
# Q8.4 - Describe history and use version as of
spark.sql("DESCRIBE HISTORY delta.`/mnt/delta/traffic_logs`").show()


+-------+-------------------+----------------+--------------------+---------+--------------------+----+------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|          userId|            userName|operation| operationParameters| job|          notebook|           clusterId|readVersion|   isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+----------------+--------------------+---------+--------------------+----+------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|      7|2025-06-19 05:14:58|7928277367239535|azuser3564_mml.lo...| OPTIMIZE|{predicate -> [],...|NULL|{2805786059712357}|0611-043414-4p180ssa|          5|SnapshotIsolation|        false|{numRemovedFiles ...|        NULL|Databricks-Runtim...|
|      6|2025-06-19 05:14:56

# 9.Advanced Conditions


In [0]:
# Q9.1 - Tag trip type based on duration
df2 = df2.withColumn("TripType", when(col("TripDurationMinutes") < 15, "Short")
                                  .when(col("TripDurationMinutes") <= 30, "Medium")
                                  .otherwise("Long"))
df2.select("LogID", "TripDurationMinutes", "TripType").show()


+-----+-------------------+--------+
|LogID|TripDurationMinutes|TripType|
+-----+-------------------+--------+
| L001|               19.0|  Medium|
| L002|               35.0|    Long|
| L003|               18.0|  Medium|
| L004|               20.0|  Medium|
| L005|               35.0|    Long|
+-----+-------------------+--------+



In [0]:
# Q9.2 - Flag vehicles with more than 3 trips in a day
from pyspark.sql.functions import to_date

df_trip_counts = df2.withColumn("TripDate", to_date("EntryTime"))\
                    .groupBy("VehicleID", "TripDate")\
                    .count()\
                    .filter("count > 3")
df_trip_counts.show()


+---------+--------+-----+
|VehicleID|TripDate|count|
+---------+--------+-----+
+---------+--------+-----+



# 10.Export & Reporting


In [0]:
# Q10.1 - Write final DataFrame to Parquet partitioned by VehicleType
df2.write.partitionBy("VehicleType").parquet("/mnt/output/traffic_parquet", mode="overwrite")


In [0]:
# Q10.2 - Write final DataFrame to CSV for dashboarding
df2.write.option("header", True).csv("/mnt/output/traffic_csv", mode="overwrite")


In [0]:
# Q10.3 - Create summary SQL View: total toll by VehicleType + ExitPoint
df2.createOrReplaceTempView("traffic_summary")
spark.sql("""
    SELECT VehicleType, ExitPoint, SUM(TollPaid) as TotalToll
    FROM traffic_summary
    GROUP BY VehicleType, ExitPoint
""").show()


+-----------+---------+---------+
|VehicleType|ExitPoint|TotalToll|
+-----------+---------+---------+
|        Car|    GateD|       50|
|      Truck|    GateC|      100|
|       Bike|    GateD|       30|
|        Bus|    GateA|       70|
|        Car|    GateC|       50|
+-----------+---------+---------+

